## 00 · Noisy input audio

Builds dataset **v3** from the v2 Kokoro clips: the same 12 questions and voices, with steady background noise at 5 dB signal-to-noise ratio (v2 has 20 dB). At this level speech recognition starts to mishear product names and codes, so the hearing stage, and how much of its error reaches the answers, shows up in the scorecard.

Run `00_input_audio` first. Clips go to `input/v3/` with a `manifest.json`; v2 stays as it is, because earlier traces reference it by path. The noise is seeded per clip, so the clips are reproducible. **Illustrative data.**

In [ ]:
import json, os, sys, wave

sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from voice_evals import config as C
from voice_evals.audio import add_noise, wav_bytes, write_bytes

SRC, OUT = C.INPUT_DIRS["v2"], C.INPUT_DIRS["v3"]
os.makedirs(OUT, exist_ok=True)
manifest = json.load(open(f"{SRC}/manifest.json"))
rows = []
for i, utterance_id in enumerate(sorted(manifest)):
    with wave.open(f"{SRC}/{utterance_id}.wav") as w:
        rate = w.getframerate()
        samples = np.frombuffer(w.readframes(w.getnframes()), dtype="<i2").astype(np.float64) / 32767
    write_bytes(f"{OUT}/{utterance_id}.wav", wav_bytes(add_noise(samples, C.NOISY_SNR_DB, seed=100 + i), rate))
    manifest[utterance_id] = {**manifest[utterance_id], "snr_db": C.NOISY_SNR_DB, "derived_from": f"{SRC}/{utterance_id}.wav"}
    rows.append((utterance_id, manifest[utterance_id]["voice"], round(len(samples) / rate, 1)))
write_bytes(f"{OUT}/manifest.json", json.dumps(manifest, indent=1).encode())
display(spark.createDataFrame(rows, "utterance_id STRING, voice STRING, seconds DOUBLE"))

In [ ]:
from IPython.display import Audio, Markdown, display as show

for utterance_id in sorted(manifest):
    show(Markdown(f"**{utterance_id}** · {manifest[utterance_id]['voice']} · {C.NOISY_SNR_DB:g} dB SNR"),
         Audio(filename=f"{OUT}/{utterance_id}.wav"))

In [ ]:
dbutils.notebook.exit(json.dumps({"input_dir": OUT, "clips": len(rows), "snr_db": C.NOISY_SNR_DB}))